In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\sgabalog\Documents\P3\Model\Retail_ABM\multi_runs\results\run_1_aggregated.csv")

In [4]:
import geopandas as gpd
import pandas as pd
import os

gpkg_path = r"c:\Users\sgabalog\Documents\P3\Model\data_local\liverpool\processed\retail_centre_type_counts.gpkg"
retail_data = gpd.read_file(gpkg_path, layer='retail_centre_counts')

amenity_cols = [
    'Foodstore', 'Personal Service', 'Professional Services', 
    'Entertainment', 'Convenience Store', 'Retail', 'Restaurant', 'Cafe'
]

# 1. Identify existing columns
existing_cols = [col for col in amenity_cols if col in retail_data.columns]
retail_data['Total_POIs'] = retail_data[existing_cols].sum(axis=1)

# 2. Automatically find the ID and Name columns
# We look for common variations of Name/ID in case they are named differently
id_col = next((c for c in ['RC_ID', 'rc_id', 'ID'] if c in retail_data.columns), None)
name_col = next((c for c in ['RC_Name', 'Name', 'name', 'retail_name'] if c in retail_data.columns), None)

# 3. Select columns safely (only including those that exist)
select_cols = [c for c in [id_col, name_col, 'Total_POIs'] if c is not None] + existing_cols
final_retail_summary = retail_data[select_cols]

print(f"Loaded {len(final_retail_summary)} centres. Columns found: {list(final_retail_summary.columns)}")
display(final_retail_summary.head())


Loaded 259 centres. Columns found: ['RC_ID', 'Total_POIs', 'Foodstore', 'Personal Service', 'Professional Services', 'Entertainment', 'Convenience Store', 'Retail', 'Restaurant', 'Cafe']


,RC_ID,Total_POIs,Foodstore,Personal Service,Professional Services,Entertainment,Convenience Store,Retail,Restaurant,Cafe
0,6442.0,15,1,3,1,0,4,1,3,2
1,7143.0,13,0,2,0,0,1,6,4,0
2,1388.0,127,11,23,16,7,10,43,8,9
3,7611.0,9,2,1,1,0,1,1,2,1
4,5054.0,22,5,5,0,0,1,8,1,2


In [5]:
final_retail_summary

,RC_ID,Total_POIs,Foodstore,Personal Service,Professional Services,Entertainment,Convenience Store,Retail,Restaurant,Cafe
0,6442.0,15,1,3,1,0,4,1,3,2
1,7143.0,13,0,2,0,0,1,6,4,0
2,1388.0,127,11,23,16,7,10,43,8,9
3,7611.0,9,2,1,1,0,1,1,2,1
4,5054.0,22,5,5,0,0,1,8,1,2
...,...,...,...,...,...,...,...,...,...,...
254,1566.0,82,1,1,15,17,0,16,21,11
255,950.0,199,4,10,24,52,2,60,30,17
256,288.0,528,7,60,164,87,10,82,66,52
257,5856.0,17,0,1,1,3,0,6,2,4


In [7]:
# Helper to clean IDs (removes .0 and whitespace)
def clean_id(x):
    s = str(x).strip()
    return s[:-2] if s.endswith('.0') else s

# 1. Clean the IDs in both datasets
results_df['RC_ID'] = results_df['RC_ID'].apply(clean_id)
final_retail_summary['RC_ID'] = final_retail_summary['RC_ID'].apply(clean_id)

# 2. Re-try the merge
full_analysis_df = results_df.merge(final_retail_summary, on='RC_ID', how='left')

# 3. Check for successful matches
matched_count = full_analysis_df['Total_POIs'].notna().sum()
print(f"Match Success: {matched_count} out of {len(full_analysis_df)} centres found their metadata.")

# Show the cleaned result
display(full_analysis_df.head())


Match Success: 239 out of 240 centres found their metadata.


C:\Users\sgabalog\AppData\Local\Temp\ipykernel_15684\962107261.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_retail_summary['RC_ID'] = final_retail_summary['RC_ID'].apply(clean_id)


,RC_ID,Total_Visits,comparison,entertainment,food_drink,grocery,drive,pt,walk,1.1,...,Avg_Utility_Modifier,Total_POIs,Foodstore,Personal Service,Professional Services,Entertainment,Convenience Store,Retail,Restaurant,Cafe
0,1000,2280671,538434.0,587179.0,402385.0,184178.0,710093.0,958704.0,611874.0,292718,...,1.000032,184.0,5.0,37.0,28.0,19.0,9.0,44.0,24.0,18.0
1,1010,2870981,646068.0,710787.0,537461.0,174655.0,926385.0,1125489.0,819107.0,352663,...,0.999991,176.0,3.0,35.0,26.0,18.0,8.0,37.0,38.0,11.0
2,1104,2566351,598403.0,584114.0,480764.0,204148.0,663254.0,703269.0,1199828.0,312435,...,1.000009,151.0,7.0,31.0,20.0,7.0,10.0,29.0,31.0,16.0
3,1164,1384880,325833.0,349747.0,272275.0,71626.0,337528.0,760343.0,287009.0,172531,...,0.999916,147.0,6.0,24.0,11.0,23.0,3.0,22.0,44.0,14.0
4,1179,2767306,538103.0,683633.0,471122.0,347139.0,929652.0,1110575.0,727079.0,320888,...,1.000010,139.0,7.0,24.0,24.0,8.0,9.0,37.0,18.0,12.0


In [9]:
full_analysis_df.to_csv(r"C:\Users\sgabalog\Documents\P3\Model\Retail_ABM\multi_runs\results\run_1_detailed.csv")


In [6]:
import pandas as pd

# 1. Load your simulation results
results_path = r"C:\Users\sgabalog\Documents\P3\Model\Retail_ABM\multi_runs\results\run_1_aggregated.csv"
results_df = pd.read_csv(results_path)

# 2. Align the ID columns for the merge
# The simulation results usually call the ID 'Retail_Centre' or it's the first column
if 'Retail_Centre' in results_df.columns:
    results_df = results_df.rename(columns={'Retail_Centre': 'RC_ID'})
else:
    # Rename the first column (the index) if it doesn't have a name
    results_df = results_df.rename(columns={results_df.columns[0]: 'RC_ID'})

# 3. Ensure both IDs are strings to prevent matching errors
results_df['RC_ID'] = results_df['RC_ID'].astype(str)
final_retail_summary['RC_ID'] = final_retail_summary['RC_ID'].astype(str)

# 4. Merge the datasets
# We use 'left' to keep all simulation results and attach the POI info where it matches
full_analysis_df = results_df.merge(final_retail_summary, on='RC_ID', how='left')

# 5. Show the results
print(f"Merged Simulation Results with Retail Metadata. Total Centres: {len(full_analysis_df)}")
display(full_analysis_df.head())


Merged Simulation Results with Retail Metadata. Total Centres: 240


C:\Users\sgabalog\AppData\Local\Temp\ipykernel_15684\2788264673.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_retail_summary['RC_ID'] = final_retail_summary['RC_ID'].astype(str)


,RC_ID,Total_Visits,comparison,entertainment,food_drink,grocery,drive,pt,walk,1.1,...,Avg_Utility_Modifier,Total_POIs,Foodstore,Personal Service,Professional Services,Entertainment,Convenience Store,Retail,Restaurant,Cafe
0,1000,2280671,538434.0,587179.0,402385.0,184178.0,710093.0,958704.0,611874.0,292718,...,1.000032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1010,2870981,646068.0,710787.0,537461.0,174655.0,926385.0,1125489.0,819107.0,352663,...,0.999991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1104,2566351,598403.0,584114.0,480764.0,204148.0,663254.0,703269.0,1199828.0,312435,...,1.000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1164,1384880,325833.0,349747.0,272275.0,71626.0,337528.0,760343.0,287009.0,172531,...,0.999916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1179,2767306,538103.0,683633.0,471122.0,347139.0,929652.0,1110575.0,727079.0,320888,...,1.000010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\sgabalog\Documents\P3\Model\Retail_ABM\outputs\visits_log_1778547436.parquet")

In [25]:
df.head(1000000)

,Day,AgentID,Postcode,Trip_Type,Retail_Centre,Grocery_Mode,Transport_Mode,Utility_Modifier,Utility_Score,Geo_Subcluster
0,1,310073,WA10 4BZ,grocery,ONLINE,online,None,1.00,0.000000,0.1
1,1,454786,CH45 2LU,grocery,ONLINE,online,None,1.00,0.000000,0.1
2,1,309854,WA10 3HT,grocery,ONLINE,online,None,1.00,0.000000,0.1
3,1,151066,L7 0JJ,grocery,ONLINE,online,None,1.00,0.000000,1.1
4,1,143450,L14 5NP,grocery,ONLINE,online,None,1.00,0.000000,1.1
...,...,...,...,...,...,...,...,...,...,...
999995,1,440707,L21 8JH,grocery,1281,bulk,pt,0.95,0.081787,1.1
999996,1,369285,PR8 3AY,grocery,1752,convenience,pt,0.95,0.772461,3.3
999997,1,445843,L20 5AR,grocery,2230,bulk,pt,0.95,0.042267,1.1
999998,1,512538,CH49 4NB,grocery,4442,convenience,walk,1.05,0.024445,0.1


In [26]:
# 2. Daily Aggregation (Visits per Day per Centre)
daily_agg = df.groupby(['Day', 'Retail_Centre']).size().reset_index(name='Visits')
daily_agg.to_csv("daily_centre_visits.csv", index=False)
print("SUCCESS: 'daily_centre_visits.csv' created.")
# 3. Total Aggregation (Total Visits per Centre, ranked)
centre_totals = df.groupby('Retail_Centre').size().reset_index(name='Total_Visits')
centre_totals = centre_totals.sort_values(by='Total_Visits', ascending=False)
centre_totals.to_csv("total_centre_rankings.csv", index=False)
print("SUCCESS: 'total_centre_rankings.csv' created.")


SUCCESS: 'daily_centre_visits.csv' created.
SUCCESS: 'total_centre_rankings.csv' created.


In [30]:
# 2. Pivot the data to get a column for every Trip Type
trip_breakdown = pd.crosstab(df['Day'], df['Trip_Type'])
# 3. Add Online and Mode totals for a complete picture
mode_breakdown = pd.crosstab(df['Day'], df['Transport_Mode'])
online_total = df[df['Grocery_Mode'] == 'online'].groupby('Day').size().rename('Online_Total')
# 4. Combine everything into one master Daily Summary
daily_master = pd.concat([trip_breakdown, mode_breakdown, online_total], axis=1).fillna(0).reset_index()


In [31]:
daily_master


,Day,comparison,entertainment,food_drink,grocery,service,drive,pt,walk,Online_Total
0,1,205722,187258,113648,308335,213352,277361,371201,267770,235582
1,2,205798,187112,113438,214833,213498,264832,355543,256735,121177
2,3,205655,187241,113690,213241,214055,265669,356236,257477,114441
3,4,205642,186923,113123,251646,213555,272081,364987,262270,150498
4,5,205575,187308,113478,250053,213857,270913,363733,262195,154663
5,6,205213,187332,113316,227337,213583,267670,359699,259544,126044
6,7,205852,187335,113566,241569,213540,268346,362852,262852,142744
7,8,205369,187115,113466,235181,213305,268187,361287,260912,134714
8,9,205562,186972,113109,241273,213431,269018,363197,260205,143180
9,10,205702,187369,113641,245979,213965,270672,364411,260641,149483
